# CSIRO Biomass - CHARMS-lite Baseline

This notebook replaces:
- frozen ResNet feature extraction
- PCA
- RandomForest

with:
- end-to-end CNN training
- multi-target biomass regression
- CHARMS-style auxiliary supervision from tabular metadata

Main outputs:
- 5 biomass targets predicted jointly
- auxiliary prediction of NDVI, height, state, and month during training
- image-only inference at test time

In [19]:
import os
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torchvision.models as models

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# Paths: update these for your Kaggle / local setup
TRAIN_CSV = "../../data/tabular/train/train.csv"
TEST_CSV = "../../data/tabular/test/test.csv"
IMAGE_FOLDER = "../../data/image"
SAMPLE_SUB_CSV = "../../data/tabular/test/test.csv"

SEED = 42
BATCH_SIZE = 16
NUM_WORKERS = 0
EPOCHS = 12
LR = 3e-4
WEIGHT_DECAY = 1e-4
IMG_SIZE = 224

TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
NUMERIC_ATTRS = ["Pre_GSHH_NDVI", "Height_Ave_cm"]
CAT_ATTRS = ["State", "month"]

DEVICE: cpu


Reproducibility

In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

Load and reshape data

In [3]:
def make_wide_train(train_csv):
    df = pd.read_csv(train_csv)

    # Keep one metadata row per image
    meta_cols = [
        "sample_id",
        "image_path",
        "Sampling_Date",
        "State",
        "Species",
        "Pre_GSHH_NDVI",
        "Height_Ave_cm",
    ]
    meta = df[meta_cols].drop_duplicates("image_path").copy()

    # Pivot targets so each image has 5 outputs
    y = (
        df.pivot_table(
            index="image_path",
            columns="target_name",
            values="target",
            aggfunc="first"
        )
        .reset_index()
    )

    out = meta.merge(y, on="image_path", how="inner")

    out["Sampling_Date"] = pd.to_datetime(out["Sampling_Date"])
    out["month"] = out["Sampling_Date"].dt.month.astype(int)

    return out


def load_test_df(test_csv):
    df = pd.read_csv(test_csv)

    # test is long format; keep one row per image
    keep_cols = ["sample_id", "image_path"]
    df_unique = df[keep_cols].drop_duplicates("image_path").copy()
    return df, df_unique


train_df = make_wide_train(TRAIN_CSV)
test_long_df, test_df = load_test_df(TEST_CSV)

print("Train images:", len(train_df))
print("Test images:", len(test_df))
train_df.head()

Train images: 357
Test images: 1


,sample_id,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g,month
0,ID1011485656__Dry_Clover_g,train/ID1011485656.jpg,2015-09-04,Tas,Ryegrass_Clover,0.62,4.6667,0.0000,31.9984,16.2751,48.2735,16.2750,9
1,ID1012260530__Dry_Clover_g,train/ID1012260530.jpg,2015-04-01,NSW,Lucerne,0.55,16.0000,0.0000,0.0000,7.6000,7.6000,7.6000,4
2,ID1025234388__Dry_Clover_g,train/ID1025234388.jpg,2015-09-01,WA,SubcloverDalkeith,0.38,1.0000,6.0500,0.0000,0.0000,6.0500,6.0500,9
3,ID1028611175__Dry_Clover_g,train/ID1028611175.jpg,2015-05-18,Tas,Ryegrass,0.66,5.0000,0.0000,30.9703,24.2376,55.2079,24.2376,5
4,ID1035947949__Dry_Clover_g,train/ID1035947949.jpg,2015-09-11,Tas,Ryegrass,0.54,3.5000,0.4343,23.2239,10.5261,34.1844,10.9605,9


Encode metadata

In [4]:
def fit_metadata_encoders(train_df):
    train_df = train_df.copy()

    # numeric scalers
    scaler = StandardScaler()
    train_df[NUMERIC_ATTRS] = scaler.fit_transform(train_df[NUMERIC_ATTRS].astype(np.float32))

    # categorical encodings
    state_vocab = {k: i for i, k in enumerate(sorted(train_df["State"].dropna().unique()))}

    train_df["State_id"] = train_df["State"].map(state_vocab).astype(int)
    train_df["month_id"] = (train_df["month"] - 1).clip(0, 11).astype(int)

    encoders = {
        "scaler": scaler,
        "state_vocab": state_vocab,
        "num_states": len(state_vocab),
        "num_months": 12,
    }
    return train_df, encoders


def apply_metadata_encoders(df, encoders):
    df = df.copy()

    # numeric
    if all(col in df.columns for col in NUMERIC_ATTRS):
        df[NUMERIC_ATTRS] = encoders["scaler"].transform(df[NUMERIC_ATTRS].astype(np.float32))
    else:
        for col in NUMERIC_ATTRS:
            df[col] = 0.0

    # categorical
    if "State" in df.columns:
        df["State_id"] = df["State"].map(encoders["state_vocab"]).fillna(0).astype(int)
    else:
        df["State_id"] = 0

    if "month" in df.columns:
        df["month_id"] = (df["month"] - 1).clip(0, 11).astype(int)
    else:
        df["month_id"] = 0

    return df


train_df, encoders = fit_metadata_encoders(train_df)
test_df = apply_metadata_encoders(test_df, encoders)

print(encoders)

{'scaler': StandardScaler(), 'state_vocab': {'NSW': 0, 'Tas': 1, 'Vic': 2, 'WA': 3}, 'num_states': 4, 'num_months': 12}


Metrics

In [5]:
def r2_score_numpy(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    ss_res = np.sum((y_true - y_pred) ** 2, axis=0)
    ss_tot = np.sum((y_true - np.mean(y_true, axis=0, keepdims=True)) ** 2, axis=0)
    r2 = 1 - ss_res / np.clip(ss_tot, 1e-12, None)
    return r2


def weighted_global_r2(y_true, y_pred):
    r2_per_target = r2_score_numpy(y_true, y_pred)
    return float(np.mean(r2_per_target))


def rmse_per_target(y_true, y_pred, target_names=TARGETS):
    out = {}
    for i, t in enumerate(target_names):
        rmse = np.sqrt(np.mean((y_true[:, i] - y_pred[:, i]) ** 2))
        out[t] = float(rmse)
    return out

Image transforms

In [6]:
train_tfms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.2),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.03),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

valid_tfms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

Dataset

In [7]:
class BiomassDataset(Dataset):
    def __init__(self, df, image_root, transforms=None, is_test=False):
        self.df = df.reset_index(drop=True).copy()
        self.image_root = image_root
        self.transforms = transforms
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def _load_image(self, rel_path):
        path = os.path.join(self.image_root, rel_path)
        img = Image.open(path).convert("RGB")
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])

        if self.transforms:
            img = self.transforms(img)

        item = {
            "image": img,
            "image_path": row["image_path"],
        }

        # auxiliary metadata
        x_num = np.array([row["Pre_GSHH_NDVI"], row["Height_Ave_cm"]], dtype=np.float32)
        x_cat = np.array([row["State_id"], row["month_id"]], dtype=np.int64)

        item["x_num"] = torch.tensor(x_num, dtype=torch.float32)
        item["x_cat"] = torch.tensor(x_cat, dtype=torch.long)

        if not self.is_test:
            y = row[TARGETS].values.astype(np.float32)
            item["targets"] = torch.tensor(y, dtype=torch.float32)

        return item

Model

In [8]:
class BiomassCharmsLite(nn.Module):
    def __init__(self, num_states, num_months=12):
        super().__init__()

        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        feat_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone

        self.dropout = nn.Dropout(0.2)

        # main regression head: 5 targets
        self.reg_head = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, len(TARGETS))
        )

        # auxiliary heads
        self.ndvi_head = nn.Linear(feat_dim, 1)
        self.height_head = nn.Linear(feat_dim, 1)
        self.state_head = nn.Linear(feat_dim, num_states)
        self.month_head = nn.Linear(feat_dim, num_months)

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.dropout(feat)

        out = {
            "y": self.reg_head(feat),
            "ndvi": self.ndvi_head(feat).squeeze(1),
            "height": self.height_head(feat).squeeze(1),
            "state": self.state_head(feat),
            "month": self.month_head(feat),
        }
        return out

Loss function

In [9]:
class CharmsLiteLoss(nn.Module):
    def __init__(self, w_num=0.30, w_cat=0.30):
        super().__init__()
        self.w_num = w_num
        self.w_cat = w_cat
        self.reg_loss = nn.SmoothL1Loss()
        self.mse = nn.MSELoss()
        self.ce = nn.CrossEntropyLoss()

    def forward(self, out, y, x_num, x_cat):
        loss_main = self.reg_loss(out["y"], y)

        loss_ndvi = self.mse(out["ndvi"], x_num[:, 0])
        loss_height = self.mse(out["height"], x_num[:, 1])
        loss_num = loss_ndvi + loss_height

        loss_state = self.ce(out["state"], x_cat[:, 0])
        loss_month = self.ce(out["month"], x_cat[:, 1])
        loss_cat = loss_state + loss_month

        loss = loss_main + self.w_num * loss_num + self.w_cat * loss_cat

        logs = {
            "loss": loss.item(),
            "loss_main": loss_main.item(),
            "loss_num": loss_num.item(),
            "loss_cat": loss_cat.item(),
        }
        return loss, logs

Train / valid functions

In [10]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running = {}

    for batch in loader:
        images = batch["image"].to(device)
        targets = batch["targets"].to(device)
        x_num = batch["x_num"].to(device)
        x_cat = batch["x_cat"].to(device)

        optimizer.zero_grad()

        out = model(images)
        loss, logs = criterion(out, targets, x_num, x_cat)

        loss.backward()
        optimizer.step()

        for k, v in logs.items():
            running[k] = running.get(k, 0.0) + v

    for k in running:
        running[k] /= max(len(loader), 1)
    return running


@torch.no_grad()
def valid_one_epoch(model, loader, criterion, device):
    model.eval()
    running = {}

    all_targets = []
    all_preds = []

    for batch in loader:
        images = batch["image"].to(device)
        targets = batch["targets"].to(device)
        x_num = batch["x_num"].to(device)
        x_cat = batch["x_cat"].to(device)

        out = model(images)
        loss, logs = criterion(out, targets, x_num, x_cat)

        for k, v in logs.items():
            running[k] = running.get(k, 0.0) + v

        all_targets.append(targets.cpu().numpy())
        all_preds.append(out["y"].cpu().numpy())

    for k in running:
        running[k] /= max(len(loader), 1)

    y_true = np.concatenate(all_targets, axis=0)
    y_pred = np.concatenate(all_preds, axis=0)

    running["weighted_r2"] = weighted_global_r2(y_true, y_pred)
    running["rmse"] = rmse_per_target(y_true, y_pred)

    return running, y_true, y_pred

Split data

In [11]:
# Group by image_path just to be safe against leakage
gkf = GroupKFold(n_splits=5)
train_idx, valid_idx = next(gkf.split(train_df, groups=train_df["image_path"]))

df_tr = train_df.iloc[train_idx].reset_index(drop=True)
df_va = train_df.iloc[valid_idx].reset_index(drop=True)

print(len(df_tr), len(df_va))

285 72


Dataloaders

In [12]:
train_ds = BiomassDataset(df_tr, IMAGE_FOLDER, transforms=train_tfms, is_test=False)
valid_ds = BiomassDataset(df_va, IMAGE_FOLDER, transforms=valid_tfms, is_test=False)
test_ds = BiomassDataset(test_df, IMAGE_FOLDER, transforms=valid_tfms, is_test=True)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

Model setup

In [13]:
model = BiomassCharmsLite(
    num_states=encoders["num_states"],
    num_months=encoders["num_months"],
).to(DEVICE)

criterion = CharmsLiteLoss(w_num=0.30, w_cat=0.30)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

print(model.__class__.__name__)

BiomassCharmsLite


Training loop

In [14]:
best_score = -1e9
best_path = "best_biomass_charms_lite.pt"
history = []

for epoch in range(EPOCHS):
    train_logs = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    valid_logs, y_true, y_pred = valid_one_epoch(model, valid_loader, criterion, DEVICE)

    row = {
        "epoch": epoch + 1,
        "train_loss": train_logs["loss"],
        "valid_loss": valid_logs["loss"],
        "valid_r2": valid_logs["weighted_r2"],
    }
    history.append(row)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  train_loss: {train_logs['loss']:.4f}")
    print(f"  valid_loss: {valid_logs['loss']:.4f}")
    print(f"  valid_r2:   {valid_logs['weighted_r2']:.4f}")
    print(f"  rmse:       {valid_logs['rmse']}")

    if valid_logs["weighted_r2"] > best_score:
        best_score = valid_logs["weighted_r2"]
        torch.save(model.state_dict(), best_path)
        print("  saved best model")

print("Best valid_r2:", best_score)

Epoch 1/12
  train_loss: 23.9840
  valid_loss: 19.2833
  valid_r2:   -0.4910
  rmse:       {'Dry_Green_g': 22.943058013916016, 'Dry_Dead_g': 18.814247131347656, 'Dry_Clover_g': 14.539497375488281, 'GDM_g': 27.241804122924805, 'Dry_Total_g': 33.00886917114258}
  saved best model
Epoch 2/12
  train_loss: 17.7851
  valid_loss: 27.5344
  valid_r2:   -2.4483
  rmse:       {'Dry_Green_g': 40.98440170288086, 'Dry_Dead_g': 27.533008575439453, 'Dry_Clover_g': 14.056879997253418, 'GDM_g': 43.4039192199707, 'Dry_Total_g': 57.53940963745117}
Epoch 3/12
  train_loss: 14.8382
  valid_loss: 13.8053
  valid_r2:   0.0395
  rmse:       {'Dry_Green_g': 20.74973487854004, 'Dry_Dead_g': 11.563997268676758, 'Dry_Clover_g': 13.954174041748047, 'GDM_g': 23.53297996520996, 'Dry_Total_g': 27.098052978515625}
  saved best model
Epoch 4/12
  train_loss: 12.7669
  valid_loss: 12.1146
  valid_r2:   0.1938
  rmse:       {'Dry_Green_g': 18.72793197631836, 'Dry_Dead_g': 11.73630428314209, 'Dry_Clover_g': 14.1500663757

Training history

In [15]:
hist_df = pd.DataFrame(history)
hist_df

,epoch,train_loss,valid_loss,valid_r2
0,1,23.984011,19.283279,-0.490992
1,2,17.785072,27.534429,-2.448312
2,3,14.838159,13.805340,0.039480
3,4,12.766874,12.114641,0.193774
4,5,11.940381,10.782928,0.319772
5,6,10.913805,11.241799,0.292701
6,7,10.377658,10.308515,0.350360
7,8,9.632088,11.125809,0.329177
8,9,9.828829,10.245354,0.369989
9,10,9.143948,9.909547,0.399678


Reload best model

In [16]:
model = BiomassCharmsLite(
    num_states=encoders["num_states"],
    num_months=encoders["num_months"],
).to(DEVICE)

model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.eval()

print("Loaded:", best_path)

Loaded: best_biomass_charms_lite.pt


C:\Users\somme\AppData\Local\Temp\ipykernel_2872\1792524022.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_path, map_location=DEVI

Predict test set

In [17]:
@torch.no_grad()
def predict_test(model, loader, device):
    model.eval()
    preds = []
    image_paths = []

    for batch in loader:
        images = batch["image"].to(device)
        out = model(images)

        preds.append(out["y"].cpu().numpy())
        image_paths.extend(batch["image_path"])

    preds = np.concatenate(preds, axis=0)
    return image_paths, preds


test_image_paths, test_preds = predict_test(model, test_loader, DEVICE)
test_preds.shape

(1, 5)

Convert predictions to submission format

In [20]:
pred_df = pd.DataFrame({
    "image_path": test_image_paths,
    "Dry_Green_g": test_preds[:, 0],
    "Dry_Dead_g": test_preds[:, 1],
    "Dry_Clover_g": test_preds[:, 2],
    "GDM_g": test_preds[:, 3],
    "Dry_Total_g": test_preds[:, 4],
})

pred_long = pred_df.melt(
    id_vars="image_path",
    value_vars=TARGETS,
    var_name="target_name",
    value_name="target_pred"
)

sample_sub = pd.read_csv(SAMPLE_SUB_CSV)

submission = sample_sub.merge(
    pred_long,
    on=["image_path", "target_name"],
    how="left"
)

submission["target"] = submission["target_pred"]
submission = submission[["sample_id", "target"]]

submission.head()

,sample_id,target
0,ID1001187975__Dry_Clover_g,3.369146
1,ID1001187975__Dry_Dead_g,25.684978
2,ID1001187975__Dry_Green_g,14.981568
3,ID1001187975__Dry_Total_g,44.746193
4,ID1001187975__GDM_g,18.737442


Save submission

In [21]:
submission.to_csv("submission_charms_lite.csv", index=False)
print("Saved submission_charms_lite.csv")

Saved submission_charms_lite.csv


sanity checks

In [22]:
print(submission.shape)
print(submission["target"].isna().sum(), "missing predictions")
submission.describe()

(5, 2)
0 missing predictions


,target
count,5.000000
mean,21.503866
std,15.303178
min,3.369146
25%,14.981568
50%,18.737442
75%,25.684978
max,44.746193
